In [69]:
import sys,os
sys.path.append(r'Z:/EnergyTrading/Python/')
sys.path.append(r'Z:/EnergyTrading/Python/Strategies/LeadLagXGB/')
from support_functions import calculate_MACD, calculate_lead_lag_triggers, calculate_regression_model_price, calc_vol_intensity_index

In [89]:
from Utilities.email_sending import send_plain_email, send_html_email

EMAIL_PASSWORD = os.getenv('EMAIL_PASSWORD') # the password needs to be set as EMAIL_PASSWORD in system variables of the computer where the process is running, it is located in S:/Algo/email_password.txt
if EMAIL_PASSWORD is None:
    raise ValueError("EMAIL_PASSWORD environment variable not set")

RECIPIENT = "zubal_andrej@energytrading.sk" # can also be a list of recipients

In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
#from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Database.DB_reader import Database
from datetime import date, timedelta

from Strategies.MultipleMarketsIntensity_class import MultiTradeIntensity as TI

from Strategies.LeadLagXGB.backtest_class import BacktestLL
from Strategies.LeadLagXGB.strategy_class import StrategyLL, VolumeClass
tol=(1e-1)/2

In [71]:
class EMA:
    def __init__(self, span):
        self.span = span
        self.value = 0
        self.alpha = 2 / (span + 1)

    def push(self, value):
        self.value = self.alpha * value + (1 - self.alpha) * self.value

class TR_class:
    def __init__(self, tau, tau_ema, burn=10):
        self.tau = tau
        self.tau_ema = tau_ema
        self.reset()
        self.__burn = burn

    @property
    def param_keys(self):
        return ['tau', 'tau_ema']

    def update_params(self, params_dict):
        if 'tau' in params_dict.keys():
            self.tau = params_dict['tau']
        if 'tau_ema' in params_dict.keys():
            self.tau_ema = params_dict['tau_ema']

    @property
    def ewma_val(self):
        return self.ewma.value

    @property
    def is_burn(self):
        return self.tot_n < self.burn

    @property
    def min_tau(self):
        return self.tau // 3

    @property
    def old_value(self):
        return self.__old_value

    @property
    def bt(self):
        return self.__bt

    @property
    def burn(self):
        return self.__burn

    @property
    def tot_n(self):
        return self.__tot_n

    def reset(self):
        self.phiT = 0
        self.ewma = EMA(self.tau_ema)
        self.ewma_T = EMA(self.tau_ema)
        self.thres = 0
        self.index = 0
        self.__bt = 1
        self.__old_value = np.nan
        self.__n = 0
        self.__tot_n = 0
        self.init = False

    def soft_reset(self):
        self.phiT = 0
        self.ewma = EMA(self.tau_ema)
        self.ewma_T = EMA(self.tau_ema)
        self.thres = 0
        self.index += 1
        self.__bt = 1
        self.__n = 0
        self.__tot_n = 0
        self.init = False

    def initialize(self):
        self.ewma.value = .5
        self.ewma_T.value = self.tau
        self.init = False

    def push(self, value, volume=None):
        if self.tot_n < 1:
            self.init = True
        else:
            diff_value = value - self.old_value
            self.__bt = self.signed_tick_vals(diff_value)
            bt = max(self.__bt, 0)
            if self.init:
                self.initialize()
                self.thres = max(abs(self.ewma.value) * self.ewma_T.value, self.min_tau)
            if self.is_burn:
                self.phiT += bt
                self.__n += 1
            elif max(self.phiT, self.__n - self.phiT) < self.thres:
                self.phiT += bt
                self.__n += 1
            else:
                self.ewma.push(self.phiT / self.__n)
                self.ewma_T.push(self.__n)
                self.phiT = 0
                self.thres = max(abs(self.ewma.value) * self.ewma_T.value, self.min_tau)
                self.index += 1
                self.__n = 0
        self.__tot_n += 1
        self.__old_value = value
        return self.index

    def signed_tick_vals(self, diff_value):
        if diff_value > 0:
            return 1
        elif diff_value < 0:
            return -1
        else:
            return 0

    def tick_imbalance_single(self, trades):
        self.soft_reset()
        index_series = []
        for trade in trades:
            value = trade[0]
            if not value or np.isnan(value):
                index_series.append((trade[2], self.index))
            else:
                index_series.append((trade[2], self.push(value)))

        return index_series

    def tick_imbalance_indices(self, trades):
        self.reset()
        index_series = []
        current_date = None
        daily_trades = []

        for trade in trades:
            trade_date = trade[2].date()
            if current_date is None:
                current_date = trade_date

            if trade_date != current_date:
                # Process the previous day's trades
                index_series.extend(self.tick_imbalance_single(daily_trades))
                daily_trades = []
                current_date = trade_date

            daily_trades.append(trade)

        # Process the last day's trades
        if daily_trades:
            index_series.extend(self.tick_imbalance_single(daily_trades))

        return index_series

def get_nine_am_unix_today_cet():
    # Define the CET timezone
    cet = pytz.timezone('CET')

    # Get today's date in the CET timezone
    today = datetime.now(cet).date()

    # Combine today's date with the time 09:00 AM in CET
    nine_am_today = cet.localize(datetime.combine(today, time(9, 0)))

    # Convert to Unix timestamp (seconds since epoch)
    unix_timestamp = int(nine_am_today.timestamp())

    return unix_timestamp

def calculate_ema(current_price, previous_ema, span):
    alpha = 2 / (span + 1)
    return alpha * current_price + (1 - alpha) * previous_ema

def get_trades_for_contract(contract, start_date, end_date):
    params_dict={}
    params_dict['tenor_list'] = ['dec'] if contract=='euadec1' else [contract[-2]]
    params_dict['tn1_list'] = [int(contract[-1])]
    params_dict['mkt_list'] = ['eua'] * len(params_dict['tenor_list']) if contract=='euadec1' else [contract[0:-2]] * len(params_dict['tenor_list'])
    params_dict['tn2_list'] = []
    params_dict['prod'] = 'base'
    params_dict['venue_list'] = ['eex']*len(params_dict['mkt_list'])
    params_dict['start_date'] = start_date
    params_dict['end_date'] = end_date
    params_dict['ns'] = 2

    # Fetch trades and best orders for the curve
    assembler = TPDataAssembly(source='trayport', user='matej')
    # assembler.set_start_end_time(start=[10,0,0], end=[12,0,0])    
    trades_dict = assembler.get_data(params_dict, target_data='trades')
    #assembler.set_data_source('database')
    #ba_dict = assembler.get_data(params_dict, target_data='best_orders')


    trades = pd.DataFrame()
    products = []
    for key in trades_dict.keys():
        trade_aux = trades_dict[key].copy()
        trade_aux.columns = [a + '_' + key for a in trade_aux.columns]
        if trades.empty:
            trades = trade_aux.copy()
        else:
            trades = pd.concat([trades, trade_aux])
        products.append(key)
    trades.sort_index(inplace=True)




    data_raw = trades
    print(data_raw.columns)
    data_raw['tradeid_'+contract]=data_raw['tradeid_'+contract].apply(lambda x: str(x)[:-7] if str(x)[-7:]==' Public' else str(x))
    df_lead = data_raw[data_raw['broker_id_'+contract]==1441][['tradeid_'+contract,'price_'+contract, 'volume_'+contract]].copy()
    
    df_lead['contract']=contract

    # data = data_raw[['price_dem1', 'volume_dem1','bidbestprice_dem1',
    #                    'askbestprice_dem1', 'mid_dem1', 'trade_side_dem1']].copy()

    df_lead.columns = [a.split('_')[0] for a in df_lead.columns]
    print(df_lead.columns)
    df_lead.columns = ['tradeid', 'trd_price', 'volume', 'contract']
    
    return df_lead

def fit_model(lead_contract, lag_contract):
    data_lead_trds = df[df['contract']==lead_contract]
    data_lag_trds = df[df['contract']==lag_contract]

    data_lead_trds['tag'] = 'lead'
    data_lag_trds['tag'] = 'lag'

    df_trds = pd.concat([data_lead_trds, data_lag_trds]).sort_index()

    df_trds['lead_price'] = df_trds[['trd_price', 'tag']].apply(lambda row: row['trd_price'] if row['tag'] == 'lead' else None, axis=1)
    df_trds['lead_volume'] = df_trds[['volume', 'tag']].apply(lambda row: row['volume'] if row['tag'] == 'lead' else None, axis=1)
    df_trds['lead_pv'] = df_trds[['trd_price', 'volume', 'tag']].apply(lambda row: row['trd_price'] * row['volume'] if row['tag'] == 'lead' else None, axis=1)

    df_trds['lag_price'] = df_trds[['trd_price', 'tag']].apply(lambda row: row['trd_price'] if row['tag'] == 'lag' else None, axis=1)
    df_trds['lag_volume'] = df_trds[['volume', 'tag']].apply(lambda row: row['volume'] if row['tag'] == 'lag' else None, axis=1)
    df_trds['lag_pv'] = df_trds[['trd_price', 'volume', 'tag']].apply(lambda row: row['trd_price'] * row['volume'] if row['tag'] == 'lag' else None, axis=1)



    agg_dict = {'index': 'first', 'datetime': 'first'}
    agg_dict.update({k: 'sum' for k in ['lead_volume', 'lead_pv', 'lag_volume', 'lag_pv']})

    df_trds['date'] = df_trds.index.date
    dates_list = sorted(list(set(df_trds['date'])))

    df_list = []
    df_list2 = []

    sort_order=[True, False, True]


    for current_day in dates_list:
        df_trds_1day = df_trds[df_trds['date'] == current_day].reset_index()
        df_trds_1day['execution_time'] = df_trds_1day['datetime'].astype('int64')  # Already in nanoseconds

        # Preparing tick data
        ti_cls = TR_class(tau=10, tau_ema=10)

        df_trds_1day = df_trds_1day.sort_values(by=['datetime', 'lag_volume', 'lag_price'], ascending=sort_order).reset_index()


        # Create the list of tuples
        lag_trades = [(row['lag_price'], row['volume'], row['execution_time']) for index, row in
                      df_trds_1day.iterrows()]


        idx_series = ti_cls.tick_imbalance_single(lag_trades)
        idx_series = pd.DataFrame(idx_series, columns=['index', 0])


        if 'level_0' in df_trds_1day.columns:
            del df_trds_1day['level_0']

        # Create returns
        df_trds_indexed_1day = pd.concat([df_trds_1day, idx_series[0]], axis=1).reset_index()
        lag_index = df_trds_indexed_1day[df_trds_indexed_1day['tag'] == 'lag'].index.min()
        lead_before_lag = df_trds_indexed_1day[(df_trds_indexed_1day['tag'] == 'lead') & (df_trds_indexed_1day.index < lag_index)]
        df_trds_indexed_1day.loc[lead_before_lag.index, 0] = 0

        df_trds_indexed_1day['tick_id'] = str(current_day) + '_' + df_trds_indexed_1day[0].fillna(0).apply(str)
        #df_trds_indexed_1day['datetime'] = df_trds_indexed_1day['timestamp']
        #df_trds_indexed_1day = df_trds_indexed_1day.set_index('datetime')
        df_trds_indexed_1day_grouped = df_trds_indexed_1day.groupby('tick_id').agg(
            agg_dict).reset_index().set_index('datetime')

        df_trds_indexed_1day_grouped['lead_price'] = df_trds_indexed_1day_grouped['lead_pv'] / \
                                                     df_trds_indexed_1day_grouped['lead_volume']
        df_trds_indexed_1day_grouped['lag_price'] = df_trds_indexed_1day_grouped['lag_pv'] / \
                                                    df_trds_indexed_1day_grouped['lag_volume']

        df_trds_indexed_1day_grouped['lead_log_ret'] = np.log(df_trds_indexed_1day_grouped['lead_price'].ffill() ).diff()
        df_trds_indexed_1day_grouped['lag_log_ret'] = np.log(df_trds_indexed_1day_grouped['lag_price'].ffill() ).diff()
        df_list.append(df_trds_indexed_1day_grouped)
        df_list2.append(df_trds_indexed_1day)

    df_trds_indexed = pd.concat(df_list).sort_index()
    df_trds = pd.concat([df_trds.sort_values(by=['datetime', 'lag_volume', 'lag_price'], ascending=sort_order).reset_index(drop=True), pd.concat(df_list2).sort_values(by=['datetime', 'lag_volume', 'lag_price'], ascending=sort_order).reset_index(drop=True)['tick_id']], axis=1)

    #Model fitting and scoring
    # Prepare to store predictions
    df_trds_indexed['lag_log_ret_pred'] = np.nan
    df_trds_indexed['coef1'] = np.nan
    df_trds_indexed['coef2'] = np.nan
    df_trds_indexed['date'] = df_trds_indexed.index.date
    dates_list = sorted(list(set(df_trds_indexed['date'])))    
    
    training_data = df_trds_indexed
    # Skip if not enough data


    # Independent variable (lead_log_ret) and dependent variable (lag_log_ret)
    X_train = training_data['lead_log_ret'].fillna(0)
    y_train = training_data['lag_log_ret'].fillna(0)

    # Add a constant to the independent variable
    X_train = sm.add_constant(X_train)

    # Fit the model using statsmodels
    model = sm.OLS(y_train, X_train).fit()
    
    return model.params[0], model.params[1], model.pvalues[0], model.pvalues[1], model.rsquared
    

In [72]:
lead_contract='dem1'
lag_contract='dem2'
start_date=(date.today()+timedelta(-14)).isoformat()
end_date=(date.today()+timedelta(-1)).isoformat()

In [73]:
start_date, end_date

('2025-04-09', '2025-04-22')

# Selecting all trades since 2024

In [74]:
# 1. Read data from source DB
conn = Database('timescaledb')

query=f"""select distinct datetime, nanotime, tradeid from  public.trades 
          where datetime>='{start_date}' and datetime<='{date.today()}'
          and instid in ('10641710', '10001075', '10100480', '10012528')
          order by datetime asc, nanotime asc""" # where rownum <= 100"""
df=conn.execute(query)


print(f"✅ Loaded {len(df)} rows from source database.")

Connected to the database timescaledb
Disconnected from the database timescaledb
✅ Loaded 106332 rows from source database.


In [75]:
df.head(-20)

,datetime,nanotime,tradeid
0,2025-04-09 07:44:29,333000000.0,5488580
1,2025-04-09 07:58:36,881000000.0,8215331
2,2025-04-09 07:59:27,323000000.0,7441265
3,2025-04-09 07:59:57,750000000.0,7441267
4,2025-04-09 08:00:07,516000000.0,7441268
...,...,...,...
106307,2025-04-22 17:57:36,988393948,Eurex T7/DEBM062025-20250422/10955/1
106308,2025-04-22 17:58:02,166985071,Eurex T7/DEBM052025-20250422/10956/1
106309,2025-04-22 17:58:09,657000000,5507135
106310,2025-04-22 17:58:16,4000000,7458896


# Selecting all lead and lag EEX trades since 2024 -> this is the base for predictor calculations

In [76]:
df_lead=get_trades_for_contract(lead_contract, start_date, end_date)
df_lag=get_trades_for_contract(lag_contract, start_date, end_date)

https://referencedata.trayport.com/instruments
Duration: 0.590s
https://analytics.trayport.com/api/trades?from=2025-04-09T06%3A00%3A00Z&until=2025-04-22T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=257&ContractType=SinglePeriod
Duration: 1.468s
Index(['tradeid_dem1', 'price_dem1', 'volume_dem1', 'action_dem1',
       'broker_id_dem1'],
      dtype='object')
Index(['tradeid', 'price', 'volume', 'contract'], dtype='object')
https://referencedata.trayport.com/instruments
Duration: 0.383s
https://analytics.trayport.com/api/trades?from=2025-04-09T06%3A00%3A00Z&until=2025-04-22T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=258&ContractType=SinglePeriod
Duration: 0.867s
Index(['tradeid_dem2', 'price_dem2', 'volume_dem2', 'action_dem2',
       'broker_id_dem2'],
      dtype='object')
Index(['tradeid', 'price', 'volume', 'contract'], dtype='object')


In [77]:
df_lag = df_lag.reset_index(drop=False)
df_lead = df_lead.reset_index(drop=False)

In [78]:
df_lag['timestamp']=df_lag['datetime']
df_lead['timestamp']=df_lead['datetime']

In [79]:
df_lead.head()

,datetime,tradeid,trd_price,volume,contract,timestamp
0,2025-04-09 08:13:59.065028906,Eurex T7/DEBM052025-20250409/99/1,61.00,1,dem1,2025-04-09 08:13:59.065028906
1,2025-04-09 08:15:02.407667875,Eurex T7/DEBM052025-20250409/101/1,61.10,1,dem1,2025-04-09 08:15:02.407667875
2,2025-04-09 08:15:11.783060789,Eurex T7/DEBM052025-20250409/102/1,61.11,1,dem1,2025-04-09 08:15:11.783060789
3,2025-04-09 08:15:22.882086992,Eurex T7/DEBM052025-20250409/103/1,61.00,1,dem1,2025-04-09 08:15:22.882086992
4,2025-04-09 08:16:00.403377295,Eurex T7/DEBM052025-20250409/109/1,61.11,1,dem1,2025-04-09 08:16:00.403377295


In [80]:
df_lag.head()

,datetime,tradeid,trd_price,volume,contract,timestamp
0,2025-04-09 08:01:02.596361637,Eurex T7/DEBM062025-20250409/1/3,62.00,3,dem2,2025-04-09 08:01:02.596361637
1,2025-04-09 08:18:03.275027514,Eurex T7/DEBM062025-20250409/133/1,63.21,1,dem2,2025-04-09 08:18:03.275027514
2,2025-04-09 08:18:03.275027514,Eurex T7/DEBM062025-20250409/133/2,63.21,2,dem2,2025-04-09 08:18:03.275027514
3,2025-04-09 08:41:59.031227112,Eurex T7/DEBM062025-20250409/367/1,62.75,1,dem2,2025-04-09 08:41:59.031227112
4,2025-04-09 08:48:46.600375652,Eurex T7/DEBM062025-20250409/437/1,63.09,1,dem2,2025-04-09 08:48:46.600375652


# Predictor calculations

## MACD

In [81]:
df_macd = df_lag.reset_index(drop=True)

In [82]:
df_macd = calculate_MACD(df_macd, 6, 14)
df_macd = calculate_MACD(df_macd, 12, 26)
df_macd = calculate_MACD(df_macd, 18, 38)
df_macd = calculate_MACD(df_macd, 30, 60)

In [83]:
df_macd = df_macd.rename(columns={'MACD_6_14': 'MACD_6_14_dem2', 'MACD_12_26': 'MACD_12_26_dem2','MACD_18_38': 'MACD_18_38_dem2','MACD_30_60': 'MACD_30_60_dem2', })
df_macd.head()

,datetime,tradeid,trd_price,volume,contract,timestamp,date,MACD_6_14_dem2,MACD_12_26_dem2,MACD_18_38_dem2,MACD_30_60_dem2
0,2025-04-09 08:01:02.596361637,Eurex T7/DEBM062025-20250409/1/3,62.00,3,dem2,2025-04-09 08:01:02.596361637,2025-04-09,0.000000,0.000000,0.000000,0.000000
1,2025-04-09 08:18:03.275027514,Eurex T7/DEBM062025-20250409/133/1,63.21,1,dem2,2025-04-09 08:18:03.275027514,2025-04-09,0.291498,0.171049,0.120409,0.073049
2,2025-04-09 08:18:03.275027514,Eurex T7/DEBM062025-20250409/133/2,63.21,2,dem2,2025-04-09 08:18:03.275027514,2025-04-09,0.291498,0.171049,0.120409,0.073049
3,2025-04-09 08:41:59.031227112,Eurex T7/DEBM062025-20250409/367/1,62.75,1,dem2,2025-04-09 08:41:59.031227112,2025-04-09,0.276608,0.190792,0.141693,0.089657
4,2025-04-09 08:48:46.600375652,Eurex T7/DEBM062025-20250409/437/1,63.09,1,dem2,2025-04-09 08:48:46.600375652,2025-04-09,0.308663,0.231209,0.177348,0.115282


## Leadlag_fair_price

In [84]:
df_ll_fair_price=calculate_regression_model_price(df_lead, df_lag, 10, 10, 3, True, 0, 0)[['datetime', 'tradeid', 'lag_price_predicted']]
df_ll_fair_price=df_ll_fair_price[df_ll_fair_price['lag_price_predicted'].isnull()==False].rename(columns={'lag_price_predicted': 'fair_price_dem2'})

In [85]:
df_ll_fair_price.head(-20)

,datetime,tradeid,fair_price_dem2
13325,2025-04-14 09:03:50.862631083,Eurex T7/DEBM052025-20250414/357/1,66.699445
13326,2025-04-14 09:03:51.234656811,Eurex T7/DEBM052025-20250414/358/1,66.699445
13327,2025-04-14 09:03:51.534586191,Eurex T7/DEBM052025-20250414/359/1,66.699445
13328,2025-04-14 09:03:51.553173780,Eurex T7/DEBM052025-20250414/360/1,66.699445
13329,2025-04-14 09:03:51.762277126,Eurex T7/DEBM052025-20250414/361/1,66.699445
...,...,...,...
30397,2025-04-22 17:54:39.887386322,Eurex T7/DEBM052025-20250422/10918/1,71.547495
30398,2025-04-22 17:55:05.204418659,Eurex T7/DEBM052025-20250422/10920/1,71.532126
30399,2025-04-22 17:56:02.081560373,Eurex T7/DEBM052025-20250422/10934/1,71.555179
30400,2025-04-22 17:56:02.096307516,Eurex T7/DEBM052025-20250422/10935/4,71.578229


## Intensity

In [86]:
df_copy=df.copy()

df_lag['tag']='lag'
df_copy['date']=df_copy['datetime'].dt.date
df_copy['nanotime'] = df_copy['nanotime'].fillna(0).apply(float)
df_copy['datetime'] = df_copy['datetime'] + pd.to_timedelta(df_copy['nanotime'], unit='ns')

df_copy=calc_vol_intensity_index(df_copy.merge(df_lag[['tradeid', 'tag', 'trd_price']], how='left', on='tradeid'), 3, 'lag')
df_copy=calc_vol_intensity_index(df_copy, 5, 'lag')
df_copy=calc_vol_intensity_index(df_copy, 7, 'lag')[['datetime', 'tradeid', 'VII_lag_3','VII_lag_5','VII_lag_7']]
df_copy.columns=['datetime', 'tradeid', 'VII_3_dem2','VII_5_dem2','VII_7_dem2']


# Saving into TimescaleDB

In [87]:
import pandas as pd
from sqlalchemy import create_engine

# Example merged dataframe (based on your given code)
df_merged = df.merge(
    df_macd[['tradeid', 'MACD_6_14_dem2', 'MACD_12_26_dem2', 'MACD_18_38_dem2', 'MACD_30_60_dem2']],
    on='tradeid', how='left'
).merge(
    df_ll_fair_price[['tradeid', 'fair_price_dem2']],
    on='tradeid', how='left'
).join(
    df_copy[['VII_3_dem2','VII_5_dem2','VII_7_dem2']]
)

# Step 1: Convert wide df to long format using melt
df_long = df_merged.melt(
    id_vars=['datetime', 'nanotime', 'tradeid'],  # Keep these columns fixed
    value_vars=['MACD_6_14_dem2', 'MACD_12_26_dem2', 'MACD_18_38_dem2', 'MACD_30_60_dem2', 'fair_price_dem2','VII_3_dem2','VII_5_dem2','VII_7_dem2'],
    var_name='pred_name',
    value_name='pred_value'
)

pred_id_mapping={
    'MACD_6_14_dem2': 'll_dem1_dem2_MACD_6_14',
    'MACD_12_26_dem2': 'll_dem1_dem2_MACD_12_26', 
    'MACD_18_38_dem2': 'll_dem1_dem2_MACD_18_38',    
    'MACD_30_60_dem2': 'll_dem1_dem2_MACD_30_60',
    'fair_price_dem2': 'll_dem1_dem2_fair_price',
    'VII_3_dem2': 'll_dem1_dem2_VII_3_dem2',
    'VII_5_dem2': 'll_dem1_dem2_VII_5_dem2',
    'VII_7_dem2': 'll_dem1_dem2_VII_7_dem2',
}

# Step 2: Add pred_name and additional columns if required
df_long['pred_id'] = df_long['pred_name'].map(pred_id_mapping)  # or use a mapping dict if needed
df_long['additional'] = None  # set accordingly if you have additional data

# Step 3: Sort by datetime for proper forward filling
df_long = df_long.sort_values(by=['datetime', 'nanotime','pred_id'], ascending=[True, True, True])

# Step 4: Forward fill within each day separately
df_long['date_only'] = df_long['datetime'].dt.date
df_long['pred_value'] = df_long.groupby(['pred_id', 'date_only'])['pred_value'].ffill()

# Optionally drop rows where pred_value is still NaN after forward fill
#df_long = df_long.dropna(subset=['pred_value'])

# Step 5: Drop helper columns
df_long = df_long.drop(columns=['date_only'])

# Locate rows where pred_value is missing and set additional message
mask_missing = df_long['pred_value'].isna()
df_long.loc[mask_missing, 'additional'] = 'Not enough observations to calculate the pred_value!'


# Optional step: enforce data types explicitly
df_long['pred_value'] = df_long['pred_value'].astype(float)

df_long=df_long[['datetime', 'nanotime', 'tradeid', 'pred_name', 'pred_id','pred_value', 
       'additional']].reset_index(drop=True)

print(len(df_long))

df_long=df_long[df_long['datetime']>=pd.to_datetime(end_date)]
# Your final DataFrame ready for DB insertion
df_long.head(-20)

850656


,datetime,nanotime,tradeid,pred_name,pred_id,pred_value,additional
753136,2025-04-22 08:04:50,595537209,Eurex T7/DEBM052025062025-20250422/12/1,MACD_12_26_dem2,ll_dem1_dem2_MACD_12_26,NaN,Not enough observations to calculate the pred_...
753137,2025-04-22 08:04:50,595537209,Eurex T7/DEBM062025-20250422/12/1,MACD_12_26_dem2,ll_dem1_dem2_MACD_12_26,0.000000,None
753138,2025-04-22 08:04:50,595537209,Eurex T7/DEBM052025-20250422/12/1,MACD_12_26_dem2,ll_dem1_dem2_MACD_12_26,0.000000,None
753139,2025-04-22 08:04:50,595537209,Eurex T7/DEBM052025062025-20250422/12/1,MACD_18_38_dem2,ll_dem1_dem2_MACD_18_38,NaN,Not enough observations to calculate the pred_...
753140,2025-04-22 08:04:50,595537209,Eurex T7/DEBM062025-20250422/12/1,MACD_18_38_dem2,ll_dem1_dem2_MACD_18_38,0.000000,None
...,...,...,...,...,...,...,...
850631,2025-04-22 17:59:55,23235887,Eurex T7/DEBM052025-20250422/10985/2,fair_price_dem2,ll_dem1_dem2_fair_price,71.707741,None
850632,2025-04-22 17:59:55,38231702,Eurex T7/DEBM052025-20250422/10986/1,MACD_12_26_dem2,ll_dem1_dem2_MACD_12_26,0.077738,None
850633,2025-04-22 17:59:55,38231702,Eurex T7/DEBM052025-20250422/10986/1,MACD_18_38_dem2,ll_dem1_dem2_MACD_18_38,0.068189,None
850634,2025-04-22 17:59:55,38231702,Eurex T7/DEBM052025-20250422/10986/1,MACD_30_60_dem2,ll_dem1_dem2_MACD_30_60,0.036411,None


In [88]:
try:# 2. Connect to TimescaleDB
    batch_size=100_000
    conn = Database('timescaledb')
    conn._connect()
    
    # 3. Insert in batches
    total_rows = len(df_long)
    for start in range(0, total_rows, batch_size):
        end = min(start + batch_size, total_rows)
        batch = df_long.iloc[start:end]
        
        batch.to_sql('dataset_entries', conn.engine,schema='public', index=False, if_exists='append',method='multi')
        print(f"✅ Inserted rows {start} to {end} into TimescaleDB.")
    
    print("🎉 All batches inserted successfully.")
    send_plain_email(
      RECIPIENT, 
    "SUCCESS: LLXGB_dem2_Predictor_calculations_for_datamart_daily_algosrv job", f'{end_date} number of records: {total_rows}',
    email_password=EMAIL_PASSWORD
)
    print('\n')

except:
    send_plain_email(
          RECIPIENT, 
        "FAIL: LLXGB_dem2_Predictor_calculations_for_datamart_daily_algosrv", f'The upload of data failed for this run, please check what is the issue.',
        email_password=EMAIL_PASSWORD
    )

Connected to the database timescaledb
✅ Inserted rows 0 to 97520 into TimescaleDB.
🎉 All batches inserted successfully.
